# Spike — verify YOLO resume=True from a mirrored last.pt

Throwaway. Proves the core assumption of the training-resume plan: after a runtime reset (local `/content` wiped), copying `last.pt` back from Drive and calling `train(resume=True)` continues mid-run instead of restarting at epoch 0.

**How to run:**
1. Run Cell 1 (setup).
2. Run Cell 2 (fresh train). **Manually STOP it** (Runtime → Interrupt execution) after ~6 epochs. Watch the `[cb]` lines — they verify the callback exposes `trainer.epoch` + `trainer.save_dir`, and mirror `last.pt` to Drive every 2 epochs.
3. **Best (true reset):** Runtime → Disconnect and delete runtime → Reconnect → re-run Cell 1 → run Cell 3.  
   **Quick (approx):** just run Cell 3 (it `rm -rf`s the local run dir to fake the reset).

**PASS** = Cell 3 logs `Resuming training ... from epoch N` (N ≈ where you stopped) and finishes.  
**FAIL** = it errors (“resume requires ...”) or restarts at epoch 0 → fall back to `project=<Drive path>` in the plan.

In [ ]:
# Cell 1 — setup
!pip install ultralytics -q
from google.colab import drive
drive.mount('/content/drive')

import shutil
from pathlib import Path

DATA       = '/content/drive/MyDrive/data check lot/data.yaml'
DET_LOCAL  = Path('/content/_det')
DET_RUN    = 'train'
DRIVE_CK   = Path('/content/drive/MyDrive/data check lot/_spike/detector')
DRIVE_LAST = DRIVE_CK / DET_RUN / 'weights' / 'last.pt'
print('data.yaml exists:', Path(DATA).exists())

In [ ]:
# Cell 2 — fresh train. >>> STOP THIS CELL MANUALLY after ~6 epochs <<<
from ultralytics import YOLO

def mirror(trainer):
    # verifies the callback contract used by the plan (#2): trainer.epoch + trainer.save_dir
    ep = int(getattr(trainer, 'epoch', -999)) + 1
    print('  [cb] epoch', ep, '| save_dir =', trainer.save_dir)
    if ep % 2:
        return
    w = Path(trainer.save_dir) / 'weights'
    dst = DRIVE_CK / DET_RUN / 'weights'
    dst.mkdir(parents=True, exist_ok=True)
    for f in ('last.pt', 'best.pt'):
        if (w / f).exists():
            shutil.copy(w / f, dst / f)
    print('  [cb] mirrored last.pt/best.pt to Drive @ epoch', ep)

m = YOLO('yolo11s.pt')
m.add_callback('on_fit_epoch_end', mirror)
m.train(data=DATA, epochs=50, imgsz=640, batch=8, project=str(DET_LOCAL), name=DET_RUN)
# After ~6 epochs: Runtime > Interrupt execution to stop here.

In [ ]:
# Cell 3 — THE TEST: simulate reset, restore last.pt from Drive, resume
# (Best: Runtime > Disconnect+delete runtime, Reconnect, re-run Cell 1, then run this.)
shutil.rmtree(DET_LOCAL, ignore_errors=True)            # fake the runtime reset (wipe local)
local_last = DET_LOCAL / DET_RUN / 'weights' / 'last.pt'
local_last.parent.mkdir(parents=True, exist_ok=True)
shutil.copy(DRIVE_LAST, local_last)
print('restored last.pt from Drive ->', local_last, '| size MB:', local_last.stat().st_size / 1e6)

from ultralytics import YOLO
m = YOLO(str(local_last))
res = m.train(resume=True)
# WATCH the log above: PASS if it says 'Resuming training ... from epoch N' (N ~ where you stopped).
print('RESUME finished:', res)